In [13]:
from io import StringIO
import numpy as np
from pathlib import Path
from Bio.PDB import MMCIFParser, PDBIO, Select, PDBParser
from Bio.SeqUtils import seq1


class _CASelect(Select):
    def accept_model(self, model):
        return model.id == 0

    def accept_chain(self, chain):
        return chain.id == "A"

    def accept_residue(self, residue):
        return residue.id[0] == " " and "CA" in residue

    def accept_atom(self, atom):
        return atom.name == "CA"


class PDBValidationError(ValueError):
    """Raised when a PDB string is not a usable CA-only structure."""


def validate_pdb(pdb_text, expected_length=None):
    if not pdb_text or not pdb_text.strip():
        raise PDBValidationError("empty PDB string")
    if not any(line.startswith(("ATOM", "HETATM")) for line in pdb_text.splitlines()):
        raise PDBValidationError("no ATOM records")

    try:
        structure = PDBParser(PERMISSIVE=False, QUIET=True).get_structure(
            "x", StringIO(pdb_text)
        )
    except Exception as e:
        raise PDBValidationError(f"cannot parse PDB: {e}") from e

    models = list(structure)
    if len(models) != 1:
        raise PDBValidationError(f"expected 1 model, found {len(models)}")
    chains = list(models[0])
    if len(chains) != 1:
        raise PDBValidationError(f"expected 1 chain, found {len(chains)}")

    coords = []
    for residue in chains[0]:
        atoms = [a.get_id() for a in residue]
        if atoms != ["CA"]:
            raise PDBValidationError(
                f"residue {residue.get_id()[1]} {residue.get_resname()} has atoms {atoms}, expected only CA"
            )
        coords.append(residue["CA"].get_coord())

    coords = np.asarray(coords, dtype=float)
    if len(coords) == 0:
        raise PDBValidationError("no residues")
    if not np.isfinite(coords).all():
        raise PDBValidationError("non-finite coordinates (nan/inf)")
    if expected_length is not None and len(coords) != expected_length:
        raise PDBValidationError(
            f"expected {expected_length} residues, found {len(coords)}"
        )
    return coords


def prepare_reference_from_cif(
    cif_path: str | Path,
    pdb_dir: str | Path,
    fasta_dir: str | Path | None = None,
) -> str:
    cif_path = Path(cif_path)
    structure = MMCIFParser(QUIET=True).get_structure(cif_path.stem, str(cif_path))
    model = structure[0]
    chain = next(model.get_chains())

    for other in list(model):
        if other is not chain:
            model.detach_child(other.id)
    chain.id = "A"

    residues = [r for r in chain if r.id[0] == " " and "CA" in r]
    for idx, res in enumerate(residues, start=1):
        res.id = (" ", idx + 100_000, " ")
    for idx, res in enumerate(residues, start=1):
        res.id = (" ", idx, " ")

    buf = StringIO()
    io = PDBIO()
    io.set_structure(structure)
    io.save(buf, select=_CASelect())
    pdb_text = buf.getvalue()

    sequence = seq1("".join(res.resname for res in residues))
    validate_pdb(pdb_text, expected_length=len(sequence))

    pdb_path = Path(pdb_dir) / f"{cif_path.stem}.pdb"
    pdb_path.parent.mkdir(parents=True, exist_ok=True)
    pdb_path.write_text(pdb_text)

    if fasta_dir is not None:
        fasta_path = Path(fasta_dir) / f"{cif_path.stem}.fasta"
        fasta_path.parent.mkdir(parents=True, exist_ok=True)
        fasta_path.write_text(f">{cif_path.stem}\n{sequence}\n")

    return sequence


def prepare_split_subset(
    cif_dir: str | Path,
    pdb_dir: str | Path,
    fasta_dir: str | Path,
    limit: int | None = 10,
) -> None:
    cif_dir = Path(cif_dir)
    pdb_dir = Path(pdb_dir)
    fasta_dir = Path(fasta_dir)
    pdb_dir.mkdir(parents=True, exist_ok=True)
    fasta_dir.mkdir(parents=True, exist_ok=True)

    cif_files = sorted(cif_dir.rglob("*.cif"))
    if not cif_files:
        raise FileNotFoundError(f"Brak plików .cif w {cif_dir}")

    ready = 0
    converted = 0
    skipped = 0
    failed = 0

    for cif_path in cif_files:
        if limit is not None and ready >= limit:
            break

        pdb_out = pdb_dir / f"{cif_path.stem}.pdb"
        fasta_out = fasta_dir / f"{cif_path.stem}.fasta"

        if (
            pdb_out.exists()
            and pdb_out.stat().st_size > 0
            and fasta_out.exists()
            and fasta_out.stat().st_size > 0
        ):
            skipped += 1
            ready += 1
            continue

        try:
            prepare_reference_from_cif(cif_path, pdb_dir, fasta_dir)
            converted += 1
            ready += 1
        except PDBValidationError as e:
            failed += 1
            print(f"[ODRZUCONO] {cif_path.name}: {e}")

    print(
        f"Summary ({cif_dir.name}): "
        f"converted={converted}, skipped={skipped}, failed={failed}"
    )

In [14]:
prepare_split_subset(
    cif_dir="../data/structures/train",
    pdb_dir="../data/pdb/train",
    fasta_dir="../data/fasta/train",
    limit=10,
)

Summary (train): converted=0, skipped=10, failed=0
